# 시간 이동 1-hop R-GAE 및 다중 인수인계 통합 우선순위

이 노트북은 저장된 평가 결과를 읽으며, 재학습은 RUN_TRAINING=True로 실행한다. 최종 실험 범위는 **10/1 학습 → 10/7 평가**, **1-hop**으로 고정한다. 모델과 중심성 값은 전체 그래프에서 먼저 계산하고, 평가 시 직접 Person–Task 후보만 추출한다. 동일 시점 결과와 2-hop 결과는 본 결론에서 제외한다.

비교 방법은 비가중 R-GAE, 역할 edge 가중 R-GAE, 역할 가중+보정 산식, Degree, Degree+보정 산식, Betweenness, Betweenness+보정 산식의 7개다. 모든 보정 방법은 구조항만 바꾸고 `구조 55 + 마감 30 + 연속성 15 − RELATED 10`을 동일하게 적용한다.

## R-GAE 도입 근거: 여러 사람의 업무를 한 수신자가 통합 인수

실제 인수인계에서는 한 사람이 여러 퇴사자·이동자의 업무를 동시에 받을 수 있다. 개인별 규칙 순위를 단순히 이어 붙이면 서로 다른 업무군 사이의 구조적 병목과 중복 관계를 비교하기 어렵다.

R-GAE는 여러 인계 대상자를 동시에 counterfactual 제거해 전체 그래프 복원오차 변화를 한 점수축으로 계산할 수 있다. 따라서 **여러 source의 업무를 한 receiver가 받을 때 통합 우선순위를 만드는 것**이 R-GAE 도입의 추가 근거가 된다. 본 시나리오는 김지수와 박보검의 업무를 원이정이 함께 인수하는 경우다. 평가 전에 원이정 edge를 추가하면 위험이 인위적으로 사라지므로 원이정은 결과의 receiver로만 기록한다.

## 역할 edge 계수와 산정 근거

10/1 그래프에서 relation-stratified masked-edge 복원 검증을 두 fold로 수행했다. 운영 GT는 사용하지 않았다.

| profile | mean | std |
|---|---|---|
| strong | 3.581 | 0.341 |
| owner_focus | 3.703 | 0.198 |
| balanced | 3.865 | 0.148 |
| mild | 3.970 | 0.392 |

평균 macro BCE가 가장 낮은 `strong` 상대비를 합계 100의 정수로 환산했다.

```text
APPROVER      31
OWNER         27
REVIEWER      21
COLLABORATOR  17
RELATED       4
합계          100
```

`APPROVER ≥ OWNER ≥ REVIEWER ≥ COLLABORATOR > RELATED`라는 책임 단조성만 사전 가정했다. 최종 산식에서 역할별 가점을 다시 더하지 않아 중복 가중을 피한다.

## 최종 우선순위 산식과 비율 근거

```text
기본 점수(100점)
= R-GAE 구조 영향도 × 55
+ 마감 긴급도 × 30
+ 업무 연속성 × 15

최종 점수 = 기본 점수 − RELATED-only × 10
```

- 구조 55: 모델 신호가 운영 항목에 종속되지 않도록 과반을 둔다.
- 마감 30: 즉시 처리 필요성을 반영하되 구조 55를 단독으로 뒤집지 못하도록 한다.
- 연속성 15: 마감이 없어도 waiting 상태나 후속 흐름이 있는 업무를 보존한다.
- RELATED −10: 업무를 제거하지 않고 최대 마감 가점 30점의 1/3만 상쇄한다. 다른 실질 역할끼리는 차등하지 않는다.

GT를 보지 않고 −5/−10/−15/−20의 순위 민감도를 비교했다.

| RELATED penalty | Spearman vs structure | Top10 Jaccard | RELATED in Top10 |
|---|---|---|---|
| 5.000 | 0.559 | 0.538 | 4.000 |
| 10.000 | 0.654 | 0.667 | 3.000 |
| 15.000 | 0.767 | 0.667 | 3.000 |
| 20.000 | 0.847 | 1.000 | 1.000 |

−10은 RELATED Top-10을 4개에서 3개로 줄이는 최소 감점이다. −15는 추가 감소가 없고 −20은 강한 역할 필터에 가까워진다. 따라서 −10을 knee point로 사용한다. 이 비율은 회귀로 학습한 정답이 아니라 명시적 운영 정책 계수다.

In [ ]:
# 완전 재현 실행은 약 2~4분 걸린다.
from pathlib import Path
import runpy

ROOT = next((p for p in (Path.cwd(), Path.cwd() / 'experiments', Path.cwd() / 'ml' / 'experiments') if (p / 'rgae_final_no_leakage_experiment.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('ml/experiments 폴더에서 노트북을 열거나 해당 폴더를 작업 디렉터리로 설정하세요.')
RUN_TRAINING = False
if RUN_TRAINING:
    runpy.run_path(str(ROOT / 'rgae_final_no_leakage_experiment.py'), run_name='__main__')
else:
    print('저장된 검증 결과를 사용합니다. 재학습하려면 RUN_TRAINING=True로 변경하세요.')

In [ ]:
import json
import pandas as pd
from IPython.display import display

RESULT_DIR = ROOT / 'results'
config = json.loads((RESULT_DIR / 'final_configuration.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(RESULT_DIR / 'final_1hop_2hop_metrics.csv')
rankings = pd.read_csv(RESULT_DIR / 'final_1hop_2hop_rankings.csv')
scenario_gt = pd.read_csv(RESULT_DIR / 'combined_kim_park_proxy_gt.csv')
scenario_metrics = pd.read_csv(RESULT_DIR / 'combined_kim_park_to_lee_metrics.csv')
scenario_rankings = pd.read_csv(RESULT_DIR / 'combined_kim_park_to_lee_rankings.csv')

display(metrics)
display(scenario_metrics)
display(scenario_gt.sort_values('combined_proxy_gt_rank'))

## 김지수 시간 이동 1-hop 결과

| method | K | TP@K | Precision@K | Recall@K | nDCG@K |
|---|---|---|---|---|---|
| temporal 10/1→10/7 | R-GAE single | 3 | 2 | 0.667 | 0.200 | 0.661 |
| temporal 10/1→10/7 | R-GAE single | 5 | 3 | 0.600 | 0.300 | 0.640 |
| temporal 10/1→10/7 | R-GAE single | 10 | 6 | 0.600 | 0.600 | 0.614 |
| temporal 10/1→10/7 | R-GAE role-weighted | 3 | 3 | 1.000 | 0.300 | 0.682 |
| temporal 10/1→10/7 | R-GAE role-weighted | 5 | 5 | 1.000 | 0.500 | 0.786 |
| temporal 10/1→10/7 | R-GAE role-weighted | 10 | 9 | 0.900 | 0.900 | 0.890 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 3 | 3 | 1.000 | 0.300 | 0.719 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 5 | 5 | 1.000 | 0.500 | 0.845 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 10 | 7 | 0.700 | 0.700 | 0.814 |
| Degree | 3 | 0 | 0.000 | 0.000 | 0.000 |
| Degree | 5 | 0 | 0.000 | 0.000 | 0.000 |
| Degree | 10 | 2 | 0.200 | 0.200 | 0.052 |
| Betweenness | 3 | 0 | 0.000 | 0.000 | 0.000 |
| Betweenness | 5 | 0 | 0.000 | 0.000 | 0.000 |
| Betweenness | 10 | 2 | 0.200 | 0.200 | 0.077 |
| Degree + corrected priority | 3 | 0 | 0.000 | 0.000 | 0.000 |
| Degree + corrected priority | 5 | 0 | 0.000 | 0.000 | 0.000 |
| Degree + corrected priority | 10 | 3 | 0.300 | 0.300 | 0.158 |
| Betweenness + corrected priority | 3 | 2 | 0.667 | 0.200 | 0.337 |
| Betweenness + corrected priority | 5 | 2 | 0.400 | 0.200 | 0.265 |
| Betweenness + corrected priority | 10 | 2 | 0.200 | 0.200 | 0.221 |

## 김지수+박보검 → 원이정 통합 시나리오

박보검에 대한 독립 human GT는 현재 데이터에 없다. 따라서 아래 통합 GT는 기존 김지수 GT에 박보검의 실질 1-hop 업무를 동일 운영 기준으로 추가한 **scenario proxy GT**다. 모델 계수 선택에는 사용하지 않았으며 독립 human-GT 성능으로 해석하지 않는다.

### 통합 proxy GT positive

| task_id | combined_proxy_gt_rank | kim_existing_gt_rank | gt_provenance |
|---|---|---|---|
| TASK-00102 | 1.000 | 1.000 | existing_kim_human_gt |
| TASK-00023 | 2.000 | 2.000 | existing_kim_human_gt |
| TASK-00066 | 3.000 |  | park_proxy_added |
| TASK-00103 | 4.000 | 3.000 | existing_kim_human_gt |
| TASK-00101 | 5.000 | 4.000 | existing_kim_human_gt |
| TASK-00015 | 6.000 |  | park_proxy_added |
| TASK-00054 | 7.000 | 5.000 | existing_kim_human_gt |
| TASK-00063 | 8.000 |  | park_proxy_added |
| TASK-00024 | 9.000 | 6.000 | existing_kim_human_gt |
| TASK-00079 | 10.000 | 7.000 | existing_kim_human_gt |
| TASK-00056 | 11.000 | 8.000 | existing_kim_human_gt |
| TASK-00080 | 12.000 | 9.000 | existing_kim_human_gt |
| TASK-00022 | 13.000 | 10.000 | existing_kim_human_gt |

### 성능

| method | K | TP@K | Precision@K | Recall@K | nDCG@K |
|---|---|---|---|---|---|
| temporal 10/1→10/7 | R-GAE single | 3 | 2 | 0.667 | 0.154 | 0.537 |
| temporal 10/1→10/7 | R-GAE single | 5 | 4 | 0.800 | 0.308 | 0.682 |
| temporal 10/1→10/7 | R-GAE single | 10 | 5 | 0.500 | 0.385 | 0.536 |
| temporal 10/1→10/7 | R-GAE role-weighted | 3 | 3 | 1.000 | 0.231 | 0.640 |
| temporal 10/1→10/7 | R-GAE role-weighted | 5 | 5 | 1.000 | 0.385 | 0.715 |
| temporal 10/1→10/7 | R-GAE role-weighted | 10 | 9 | 0.900 | 0.692 | 0.744 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 3 | 3 | 1.000 | 0.231 | 0.639 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 5 | 5 | 1.000 | 0.385 | 0.759 |
| temporal 10/1→10/7 | R-GAE role-weighted + corrected priority | 10 | 8 | 0.800 | 0.615 | 0.759 |
| Degree | 3 | 0 | 0.000 | 0.000 | 0.000 |
| Degree | 5 | 0 | 0.000 | 0.000 | 0.000 |
| Degree | 10 | 1 | 0.100 | 0.077 | 0.076 |
| Betweenness | 3 | 0 | 0.000 | 0.000 | 0.000 |
| Betweenness | 5 | 0 | 0.000 | 0.000 | 0.000 |
| Betweenness | 10 | 2 | 0.200 | 0.154 | 0.060 |
| Degree + corrected priority | 3 | 1 | 0.333 | 0.077 | 0.422 |
| Degree + corrected priority | 5 | 1 | 0.200 | 0.077 | 0.325 |
| Degree + corrected priority | 10 | 2 | 0.200 | 0.154 | 0.259 |
| Betweenness + corrected priority | 3 | 2 | 0.667 | 0.154 | 0.293 |
| Betweenness + corrected priority | 5 | 2 | 0.400 | 0.154 | 0.225 |
| Betweenness + corrected priority | 10 | 3 | 0.300 | 0.231 | 0.259 |

In [ ]:
# 통합 시나리오의 실제 Top-10과 GT 열 확인
METHOD = 'temporal 10/1→10/7 | R-GAE role-weighted + corrected priority'
columns = [
    'rank', 'task_id', 'task_name', 'source_people', 'receiver',
    'combined_proxy_gt_label', 'combined_proxy_gt_rank', 'gt_provenance',
    'model_score', 'deadline_score', 'workflow_score', 'related_only', 'final_score'
]
display(scenario_rankings[scenario_rankings['method'] == METHOD][columns].head(10))

## 결론

- 김지수 단독 시간 이동 1-hop에서 역할 가중 R-GAE는 @10 9/10, 보정 산식은 @3·@5 순서를 개선하고 @10 7/10이다.
- Degree+보정은 @10 3/10, Betweenness+보정은 @10 2/10으로 운영 항목을 동일하게 추가해도 R-GAE+보정의 7/10에 미치지 못한다.
- 김지수+박보검 통합 시나리오에서 역할 가중 R-GAE는 proxy GT 기준 @10 9/10, 보정 산식은 8/10이다.
- 구조 위험 순위는 회수율 확인용, 보정 우선순위는 실제 처리 순서용으로 함께 제공하는 것이 적절하다.
- 통합 시나리오를 정식 성능으로 보고하려면 박보검 업무의 human GT를 별도로 확정해야 한다.